In [31]:
import numpy as np

# making a dataset
texts = [
    "I love this movie",
    "It was fantastic and great",
    "Best film ever",
    "Absolutely amazing show",
    "Worst movie ever",
    "I hated it",
    "Terrible film and acting",
    "Very boring and waste of time"
]

# Labels: 1 = Positive, 0 = Negative
labels = np.array([1, 1, 1, 1, 0, 0, 0, 0], dtype=np.float32)

In [32]:
print("Dataset size:", len(texts))
print(texts)

Dataset size: 8
['I love this movie', 'It was fantastic and great', 'Best film ever', 'Absolutely amazing show', 'Worst movie ever', 'I hated it', 'Terrible film and acting', 'Very boring and waste of time']


In [33]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

In [34]:
vectorizer = TextVectorization()
vectorizer.adapt(texts)
x = vectorizer(texts).numpy()

In [35]:
x

array([[ 5, 17, 13,  3,  0,  0],
       [ 4, 10, 20,  2, 19,  0],
       [22,  6,  7,  0,  0,  0],
       [25, 23, 15,  0,  0,  0],
       [ 8,  3,  7,  0,  0,  0],
       [ 5, 18,  4,  0,  0,  0],
       [14,  6,  2, 24,  0,  0],
       [11, 21,  2,  9, 16, 12]])

### Making the GRU model

In [37]:
from tensorflow.keras import Sequential, layers
from tensorflow.keras.layers import Embedding, GRU, Dense

In [38]:
vocab_size = len(vectorizer.get_vocabulary())
sequence_length = x.shape[1]

In [40]:
model = Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=8, input_shape=(sequence_length,)),
    layers.GRU(units=16),
    layers.Dense(units=1, activation='sigmoid')
])

In [41]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 6, 8)           │           208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 16)             │         1,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,473 (5.75 KB)

 Trainable params: 1,473 (5.75 KB)

 Non-trainable params: 0 (0.00 B)

In [42]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
history = model.fit(x, labels, epochs=20, verbose=1)

Epoch 1/20


I0000 00:00:1782843566.586579     646 cuda_dnn.cc:529] Loaded cuDNN version 91002


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.3750 - loss: 0.6950
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.6250 - loss: 0.6946
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.3750 - loss: 0.6942
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.3750 - loss: 0.6939
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.6250 - loss: 0.6935
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.5000 - loss: 0.6932
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.6250 - loss: 0.6928
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.7500 - loss: 0.6925
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 1.0000 - loss: 0.6921
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.8750 - loss: 0.6918
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.8750 - loss: 0.6914
Epoch 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.7500 - loss: 0.6910
Epoch 13/20
1/

### Making Prediction

In [44]:
test_reviews = [
    "great movie",
    "terrible show"
]

# 1. Vectorize the test text using the same vectorizer
X_test = vectorizer(test_reviews).numpy()

# 2. Make predictions
predictions = model.predict(X_test)

# 3. Print the results
for i, review in enumerate(test_reviews):
    sentiment = "Positive" if predictions[i][0] > 0.5 else "Negative"
    print(f"Review: '{review}' -> Sentiment Score: {predictions[i][0]:.4f} ({sentiment})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Review: 'great movie' -> Sentiment Score: 0.4979 (Negative)
Review: 'terrible show' -> Sentiment Score: 0.5033 (Positive)
